In [1]:
# Heart Disease Model Comparison Notebook

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load preprocessed dataset
# Assume df is cleaned and has 'target' column
from data_utils import load_clean_data

df = load_clean_data('../data/processed/clean_heart.csv')

X = df.drop(columns='target')
y = df['target']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Helper to evaluate model
results = {}

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    roc_auc = roc_auc_score(y_test, y_prob)
    results[name] = {
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'roc_auc': roc_auc,
        'conf_matrix': confusion_matrix(y_test, y_pred),
        'fpr': roc_curve(y_test, y_prob)[0],
        'tpr': roc_curve(y_test, y_prob)[1],
    }
    print(f"\n{name} Performance:")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc)

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
evaluate_model("Logistic Regression (default)", log_reg, X_test, y_test)

param_grid_lr = {'C': [0.01, 0.1, 1, 10]}
gs_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_grid_lr, cv=cv, scoring='roc_auc')
gs_lr.fit(X_train, y_train)
evaluate_model("Logistic Regression (tuned)", gs_lr.best_estimator_, X_test, y_test)

# 2. SVM
svm = SVC(probability=True)
svm.fit(X_train, y_train)
evaluate_model("SVM (default)", svm, X_test, y_test)

param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
gs_svm = GridSearchCV(SVC(probability=True), param_grid_svm, cv=cv, scoring='roc_auc')
gs_svm.fit(X_train, y_train)
evaluate_model("SVM (tuned)", gs_svm.best_estimator_, X_test, y_test)

# 3. Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
evaluate_model("Random Forest (default)", rf, X_test, y_test)

param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [None, 5, 10]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=cv, scoring='roc_auc')
gs_rf.fit(X_train, y_train)
evaluate_model("Random Forest (tuned)", gs_rf.best_estimator_, X_test, y_test)

# 4. XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
evaluate_model("XGBoost (default)", xgb, X_test, y_test)

param_grid_xgb = {'n_estimators': [100, 200], 'max_depth': [3, 5, 7]}
gs_xgb = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'), param_grid_xgb, cv=cv, scoring='roc_auc')
gs_xgb.fit(X_train, y_train)
evaluate_model("XGBoost (tuned)", gs_xgb.best_estimator_, X_test, y_test)

# 5. Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
evaluate_model("Naive Bayes (default)", nb, X_test, y_test)

# Summary Table
summary_df = pd.DataFrame({k: {'ROC-AUC': v['roc_auc']} for k, v in results.items()}).T
summary_df = summary_df.sort_values(by='ROC-AUC', ascending=False)
print("\nSummary Table:")
print(summary_df)

# ROC Curve Plot
plt.figure(figsize=(10, 7))
for name, res in results.items():
    plt.plot(res['fpr'], res['tpr'], label=f"{name} (AUC={res['roc_auc']:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'xgboost'